# GlowWise Skincare - Modelagem ML

Este notebook documenta a etapa de modelagem do Desafio 02. A proposta é transformar os produtos coletados pelo bot em uma decisão de recomendação: *vale_comprar* ou *nao_vale_comprar*.

A mesma lógica operacional está no script *ml/treinar_modelo.py*, que deve ser usado para retreinar o modelo fora do notebook.

## Objetivo da Modelagem

- Usar o dataset processado pelo bot de coleta.
- Criar um rótulo supervisionado inicial para *vale_comprar*.
- Treinar um pipeline scikit-learn com **Pipeline** e **ColumnTransformer**.
- Comparar 3 modelos diferentes no MLflow.
- Promover o melhor modelo para o MLflow Registry com alias **@production**.
- Testar uma predição que será consumida pela FastAPI local.

In [ ]:
from pathlib import Path
import sys

import mlflow
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR / 'ml'))

from treinar_modelo import ( 
    DATASET_PATH,
    EXPERIMENT_NAME,
    FEATURES,
    MLFLOW_TRACKING_URI,
    ML_DATASET_PATH,
    MODEL_NAME,
    TARGET,
    avaliar,
    criar_pipeline,
    modelos_candidatos,
    preparar_dataset,
    promover_modelo,
)

print(PROJECT_DIR)
print(DATASET_PATH)

C:\Users\rebec\Documents\GlowWise Skincare
C:\Users\rebec\Documents\GlowWise Skincare\data\processed\produtos_sanitizados.csv


## 1. Carregamento e Preparação do Dataset

O dataset vem de data/processed/produtos_sanitizados.csv, gerado a partir do *coleta.json*. A funcao preparar_dataset cria:

- **texto_produto**: texto combinado com produto, marca, termo e descricao original;
- **sinal_promocional**: indicador simples de termos como outlet, cupom e desconto;
- **vale_comprar**: rotulo inicial usado no treino.

In [2]:
df = preparar_dataset(DATASET_PATH)
df.to_csv(ML_DATASET_PATH, index=False, encoding='utf-8-sig')

print(df.shape)
df[[
    'produto',
    'marca',
    'categoria_busca',
    'loja',
    'preco',
    'sinal_promocional',
    TARGET,
]].head(10)

(38, 16)


,produto,marca,categoria_busca,loja,preco,sinal_promocional,vale_comprar
0,Niacinamide Gel Concentrado - Gel de Limpeza F...,Ada Tina,limpeza,Beleza na Web,170.79,0,0
1,Cleanance Aqua-Gel - Hidratante Facial 30g,Avène,hidratacao,Beleza na Web,148.90,1,1
2,Botik Vitamina C - Gel de Limpeza Facial Antio...,O Boticário,limpeza,Beleza na Web,87.90,0,0
3,Botik Vitamina C - Gel de Limpeza Facial Antio...,O Boticário,limpeza,Beleza na Web,29.90,0,1
4,Niacinamida 10% + Zinco 1% - Sérum Hidratante ...,Carla Leone,hidratacao,Beleza na Web,79.90,0,1
5,Niacinamida + Zinco - Gel de Limpeza Facial 60ml,Carla Leone,limpeza,Beleza na Web,49.90,0,1
6,Acne Control - Gel de Limpeza Facial 340g,Cerave,limpeza,Beleza na Web,105.90,1,1
7,Acne Control Refil - Gel de Limpeza Facial 227g,Cerave,limpeza,Beleza na Web,65.90,1,1
8,Peles Equilibradas a Oleosas - Gel de Limpeza ...,Cerave,limpeza,Beleza na Web,69.90,0,1
9,Vitamina C - Sérum Facial 30g,Creamy,hidratacao,Beleza na Web,150.00,1,1


In [3]:
df[TARGET].value_counts(normalize=False).rename({0: 'nao_vale_comprar', 1: 'vale_comprar'})

vale_comprar
vale_comprar        25
nao_vale_comprar    13
Name: count, dtype: int64

## 2. Features e Rotulo

As features usadas pelo modelo são:

- **texto_produto**: vetorizado com TF-IDF;
- **categoria_busca** e **loja**: codificadas com OneHotEncoder;
- **preco**: padronizado com StandardScaler.

O alvo é *vale_comprar*.

In [4]:
from sklearn.model_selection import train_test_split

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print('Treino:', X_train.shape)
print('Teste:', X_test.shape)

Treino: (28, 4)
Teste: (10, 4)


## 3. Comparativo de Modelos no MLflow

O desafio exige pelo menos 3 algoritmos distintos. Aqui comparamos:

- Logistic Regression;
- Random Forest;
- Gradient Boosting.

A metrica principal escolhida é f1, pois equilibra precision e recall para a classe de recomendação.

In [5]:
import mlflow.sklearn

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

resultados = []
melhor = None

for nome, modelo in modelos_candidatos().items():
    pipeline = criar_pipeline(modelo)

    with mlflow.start_run(run_name=f'notebook_{nome}') as run:
        pipeline.fit(X_train, y_train)
        predicoes = pipeline.predict(X_test)
        metricas = avaliar(y_test, predicoes)

        mlflow.log_params({
            'algoritmo': nome,
            'origem': 'notebook_02_modelagem_ml',
            'dataset': str(DATASET_PATH.relative_to(PROJECT_DIR)),
            'total_registros': int(len(df)),
            'features': ','.join(FEATURES),
            'rotulo': TARGET,
        })
        mlflow.log_metrics(metricas)
        mlflow.log_artifact(str(ML_DATASET_PATH), artifact_path='dataset')
        mlflow.sklearn.log_model(
            sk_model=pipeline,
            name='modelo',
            registered_model_name=MODEL_NAME,
            input_example=X_train.head(3),
        )

        resultado = {'run_id': run.info.run_id, 'modelo': nome, **metricas}
        resultados.append(resultado)

        if melhor is None or metricas['f1'] > melhor['f1']:
            melhor = resultado

pd.DataFrame(resultados).sort_values('f1', ascending=False)

2026/06/07 16:34:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


C:\Users\rebec\Documents\GlowWise Skincare\.ml-glowwise\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Registered model 'glowwise-vale-comprar' already exists. Creating a new version of this model...
Created version '4' of model 'glowwise-vale-comprar'.


2026/06/07 16:35:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


C:\Users\rebec\Documents\GlowWise Skincare\.ml-glowwise\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Registered model 'glowwise-vale-comprar' already exists. Creating a new version of this model...
Created version '5' of model 'glowwise-vale-comprar'.


2026/06/07 16:35:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


C:\Users\rebec\Documents\GlowWise Skincare\.ml-glowwise\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Registered model 'glowwise-vale-comprar' already exists. Creating a new version of this model...
Created version '6' of model 'glowwise-vale-comprar'.


,run_id,modelo,accuracy,precision,recall,f1
0,cf1023cafd234f5a8c660e35ccbc5a7c,logistic_regression,1.0,1.00,1.000000,1.000000
1,9004944cca4548d9b585e35627a180cc,random_forest,0.8,1.00,0.714286,0.833333
2,c02fd696a6634433ab8454c6c4f6bd06,gradient_boosting,0.7,0.75,0.857143,0.800000


## 4. Promoção do Melhor Modelo

O melhor modelo é promovido no MLflow Registry com alias *@production*. A FastAPI carrega exatamente esse alias usando models:/glowwise-vale-comprar@production.

In [6]:
print(melhor)
promover_modelo(melhor['run_id'], MODEL_NAME)

{'run_id': 'cf1023cafd234f5a8c660e35ccbc5a7c', 'modelo': 'logistic_regression', 'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}
Modelo promovido: glowwise-vale-comprar versao 4 alias @production


## 5. Teste de Predição Local

Este teste simula o payload que a FastAPI recebe em *POST /predict*.

In [7]:
modelo_producao = mlflow.pyfunc.load_model(f'models:/{MODEL_NAME}@production')

amostra = pd.DataFrame([
    {
        'texto_produto': 'Gel de Limpeza Facial Vitamina C 120g Garnier gel de limpeza facial vitamina c Outlet por R$48,90',
        'categoria_busca': 'limpeza',
        'loja': 'Beleza na Web',
        'preco': 48.90,
    }
])

predicao = modelo_producao.predict(amostra)
classe = 'vale_comprar' if int(predicao[0]) == 1 else 'nao_vale_comprar'
classe

'vale_comprar'

## 6. Como Executar Fora do Notebook

Treino:

```powershell
.\.ml-glowwise\Scripts\python.exe ml\treinar_modelo.py
```

API:

```powershell
.\.ml-glowwise\Scripts\python.exe -m uvicorn api.main:app --reload
```

Endpoints:

- GET http://127.0.0.1:8000/saude
- POST http://127.0.0.1:8000/predict